# 🚗 Car Segmentation — Инференс одной RGB модели

Одна модель обучена на RGB с consistency loss (видела LAB+HSV при обучении).
Экспериментируем: подаём на вход разные цветовые пространства и усредняем маски.

## 0. Настройка путей

In [ ]:
import sys

PROJECT_ROOT  = r"D:\ML_2\Carvana\code\ver3\car-segmentation"
WEIGHTS_PATH  = rf"{PROJECT_ROOT}\checkpoints\rgb\best_model.pth"
IMG_SIZE      = 512

sys.path.insert(0, PROJECT_ROOT)
print("Готово!")

## 1. Импорты

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import torchvision.transforms.v2 as tfs_v2
from PIL import Image
from itertools import combinations

from src.model import UNetModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Вспомогательные функции

In [ ]:
# ============================================================
#  Цветовые пространства и нормализации
# ============================================================

COLORSPACES = {
    "rgb":   {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
    "lab":   {"mean": [0.5, 0.5, 0.5],       "std": [0.5, 0.5, 0.5]},
    "hsv":   {"mean": [0.5, 0.5, 0.5],       "std": [0.5, 0.5, 0.5]},
    "ycrcb": {"mean": [0.5, 0.5, 0.5],       "std": [0.5, 0.5, 0.5]},
    "hls":   {"mean": [0.5, 0.5, 0.5],       "std": [0.5, 0.5, 0.5]},
}

CV2_CONVERSIONS = {
    "rgb":   None,
    "lab":   cv2.COLOR_RGB2LAB,
    "hsv":   cv2.COLOR_RGB2HSV,
    "ycrcb": cv2.COLOR_RGB2YCrCb,
    "hls":   cv2.COLOR_RGB2HLS,
}


def convert_image(img_rgb: Image.Image, colorspace: str) -> Image.Image:
    """Конвертирует PIL RGB в нужное цветовое пространство"""
    if colorspace == "rgb":
        return img_rgb
    img_np = np.array(img_rgb)
    converted = cv2.cvtColor(img_np, CV2_CONVERSIONS[colorspace])
    return Image.fromarray(converted)


def get_transform(colorspace: str) -> tfs_v2.Compose:
    norm = COLORSPACES[colorspace]
    return tfs_v2.Compose([
        tfs_v2.Resize((IMG_SIZE, IMG_SIZE)),
        tfs_v2.ToImage(),
        tfs_v2.ToDtype(torch.float32, scale=True),
        tfs_v2.Normalize(mean=norm["mean"], std=norm["std"]),
    ])


# ============================================================
#  Загрузка модели
# ============================================================

def load_model(weights_path: str) -> UNetModel:
    model = UNetModel(pretrained=False)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval().to(device)
    return model


# ============================================================
#  Предсказание одного входа с TTA
# ============================================================

def predict_single(model, tensor: torch.Tensor, use_tta: bool = True) -> torch.Tensor:
    """Предсказание одного тензора. Возвращает вероятности [0,1]"""
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor.to(device)))
        if use_tta:
            flipped = torch.flip(tensor, dims=[3])
            pred_f  = torch.sigmoid(model(flipped.to(device)))
            pred_f  = torch.flip(pred_f, dims=[3])
            pred    = (pred + pred_f) / 2
    return pred.cpu()


# ============================================================
#  Постобработка
# ============================================================

def remove_noise(mask_np: np.ndarray, min_area_ratio: float) -> np.ndarray:
    total = mask_np.shape[0] * mask_np.shape[1]
    min_area = int(total * min_area_ratio)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_np, connectivity=8)
    clean = np.zeros_like(mask_np)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == i] = 255
    return clean


def to_binary_mask(prob_tensor: torch.Tensor,
                   threshold: float = 0.5,
                   min_area_ratio: float = 0.01) -> np.ndarray:
    """Тензор вероятностей → бинарная маска uint8 [H, W]"""
    t = prob_tensor
    while t.dim() > 2:
        t = t.squeeze(0)
    mask = (t.numpy() * 255).astype(np.uint8)
    _, binary = cv2.threshold(mask, int(threshold * 255), 255, cv2.THRESH_BINARY)
    if min_area_ratio > 0:
        binary = remove_noise(binary, min_area_ratio)
    return binary


print("Функции готовы!")

## 3. Загрузка модели

In [ ]:
model = load_model(WEIGHTS_PATH)
print(f"Модель загружена: {WEIGHTS_PATH}")

## 4. Получение масок для всех цветовых пространств

In [ ]:
def get_all_masks(img_path: str, use_tta: bool = True) -> dict:
    """
    Прогоняет одно изображение через модель во всех цветовых пространствах.
    Возвращает словарь {colorspace: prob_tensor}
    """
    img_rgb = Image.open(img_path).convert("RGB")
    img_display = np.array(img_rgb.resize((IMG_SIZE, IMG_SIZE)))

    predictions = {}
    converted   = {}

    for cs in COLORSPACES:
        img_cs = convert_image(img_rgb, cs)
        tf     = get_transform(cs)
        tensor = tf(img_cs).unsqueeze(0)
        predictions[cs] = predict_single(model, tensor, use_tta)
        converted[cs]   = np.array(img_cs.resize((IMG_SIZE, IMG_SIZE)))

    return {
        "img_rgb":   img_display,
        "converted": converted,
        "preds":     predictions,
    }


def ensemble(
    masks_dict: dict,
    colorspaces: list,
    weights: list = None,
    threshold: float = 0.5,
    min_area_ratio: float = 0.01,
) -> dict:
    """
    Взвешенно усредняет маски выбранных цветовых пространств.

    colorspaces: список пространств для усреднения, например ["rgb", "lab", "hsv"]
    weights:     список весов, если None — равные веса
    """
    preds = masks_dict["preds"]
    if weights is None:
        weights = [1.0 / len(colorspaces)] * len(colorspaces)

    total_w = sum(weights)
    avg = sum(preds[cs] * (w / total_w) for cs, w in zip(colorspaces, weights))

    clean = to_binary_mask(avg.squeeze(0), threshold, min_area_ratio)
    car   = masks_dict["img_rgb"].copy()
    car[clean < 127] = 0

    return {"prob": avg, "mask": clean, "car_only": car}


print("Функции get_all_masks и ensemble готовы!")

## 5. Предсказание на картинке

In [ ]:
# ← Укажи путь к картинке
IMG_PATH = r"D:\ML_2\Carvana\train\00087a6bd4dc_01.jpg"

data = get_all_masks(IMG_PATH, use_tta=True)
print("Маски получены для:", list(data["preds"].keys()))

## 6. Визуализация всех входов и масок

In [ ]:
cs_list = list(COLORSPACES.keys())
n = len(cs_list)

fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))

for i, cs in enumerate(cs_list):
    # Верхний ряд — входное изображение
    axes[0, i].imshow(data["converted"][cs])
    axes[0, i].set_title(f"Вход: {cs.upper()}")
    axes[0, i].axis("off")

    # Нижний ряд — маска
    mask = to_binary_mask(data["preds"][cs].squeeze(0), threshold=0.5)
    axes[1, i].imshow(mask, cmap="gray")
    axes[1, i].set_title(f"Маска {cs.upper()}")
    axes[1, i].axis("off")

plt.suptitle("Одна модель — разные входы", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Эксперимент: одно пространство vs ансамбль нескольких

In [ ]:
# Сравниваем все одиночные входы + разные комбинации
variants = [
    (["rgb"],              [1.0],           "Только RGB"),
    (["lab"],              [1.0],           "Только LAB"),
    (["hsv"],              [1.0],           "Только HSV"),
    (["ycrcb"],            [1.0],           "Только YCrCb"),
    (["hls"],              [1.0],           "Только HLS"),
    (["rgb", "lab"],       [0.6, 0.4],      "RGB + LAB"),
    (["rgb", "hsv"],       [0.6, 0.4],      "RGB + HSV"),
    (["rgb", "lab","hsv"], [0.5, 0.3, 0.2], "RGB+LAB+HSV"),
    (list(COLORSPACES),   None,             "Все 5 (равные)"),
]

n = len(variants)
fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))

for i, (cs_list_v, wts, label) in enumerate(variants):
    res = ensemble(data, cs_list_v, wts, threshold=0.5, min_area_ratio=0.01)
    axes[0, i].imshow(res["mask"], cmap="gray")
    axes[0, i].set_title(label, fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(res["car_only"])
    axes[1, i].axis("off")

plt.suptitle("Сравнение комбинаций входов", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Эксперимент: подбор весов для RGB + LAB + HSV

In [ ]:
# Меняем вес RGB от 0.3 до 0.8, остальное делим поровну между LAB и HSV
rgb_weights = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

fig, axes = plt.subplots(2, len(rgb_weights), figsize=(4 * len(rgb_weights), 8))

for i, w_rgb in enumerate(rgb_weights):
    w_rest = (1 - w_rgb) / 2
    res = ensemble(data,
                   ["rgb", "lab", "hsv"],
                   [w_rgb, w_rest, w_rest],
                   threshold=0.5, min_area_ratio=0.01)
    axes[0, i].imshow(res["mask"], cmap="gray")
    axes[0, i].set_title(f"RGB={w_rgb}\nLAB=HSV={w_rest:.2f}", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(res["car_only"])
    axes[1, i].axis("off")

plt.suptitle("Подбор веса RGB", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Эксперимент: разные пороги

In [ ]:
# ← Выбери лучшую комбинацию из эксперимента выше
BEST_CS  = ["rgb", "lab", "hsv"]
BEST_WTS = [0.5, 0.3, 0.2]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

fig, axes = plt.subplots(2, len(thresholds), figsize=(4 * len(thresholds), 8))

for i, thr in enumerate(thresholds):
    res = ensemble(data, BEST_CS, BEST_WTS, threshold=thr, min_area_ratio=0.01)
    axes[0, i].imshow(res["mask"], cmap="gray")
    axes[0, i].set_title(f"threshold={thr}")
    axes[0, i].axis("off")
    axes[1, i].imshow(res["car_only"])
    axes[1, i].axis("off")

plt.suptitle("Влияние порога бинаризации", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Эксперимент: удаление шума

In [ ]:
area_ratios = [0.0, 0.001, 0.005, 0.01, 0.03, 0.05]

fig, axes = plt.subplots(2, len(area_ratios), figsize=(4 * len(area_ratios), 8))

for i, ratio in enumerate(area_ratios):
    res = ensemble(data, BEST_CS, BEST_WTS, threshold=0.5, min_area_ratio=ratio)
    axes[0, i].imshow(res["mask"], cmap="gray")
    axes[0, i].set_title(f"min_area={ratio}")
    axes[0, i].axis("off")
    axes[1, i].imshow(res["car_only"])
    axes[1, i].axis("off")

plt.suptitle("Влияние удаления шума", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Финальный результат с лучшими параметрами

In [ ]:
# ← Заполни лучшие параметры после экспериментов выше
FINAL_CS        = ["rgb", "lab", "hsv"]
FINAL_WEIGHTS   = [0.5, 0.3, 0.2]
FINAL_THRESHOLD = 0.5
FINAL_MIN_AREA  = 0.01

res = ensemble(data, FINAL_CS, FINAL_WEIGHTS, FINAL_THRESHOLD, FINAL_MIN_AREA)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(data["img_rgb"])
axes[0].set_title("Оригинал")
axes[0].axis("off")
axes[1].imshow(res["mask"], cmap="gray")
axes[1].set_title(f"Маска ({'+'.join(FINAL_CS)})")
axes[1].axis("off")
axes[2].imshow(res["car_only"])
axes[2].set_title("Машина без фона")
axes[2].axis("off")
plt.tight_layout()
plt.savefig("result.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: result.png")

## 12. Пакетная обработка

In [ ]:
IMAGE_PATHS = [
    r"D:\ML_2\Carvana\train\00087a6bd4dc_01.jpg",
    r"D:\ML_2\Carvana\train\00087a6bd4dc_02.jpg",
    r"D:\ML_2\Carvana\train\00087a6bd4dc_03.jpg",
]

n = len(IMAGE_PATHS)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, path in enumerate(IMAGE_PATHS):
    d   = get_all_masks(path, use_tta=True)
    res = ensemble(d, FINAL_CS, FINAL_WEIGHTS, FINAL_THRESHOLD, FINAL_MIN_AREA)
    axes[i, 0].imshow(d["img_rgb"])
    axes[i, 0].set_title(os.path.basename(path))
    axes[i, 0].axis("off")
    axes[i, 1].imshow(res["mask"], cmap="gray")
    axes[i, 1].set_title("Маска")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(res["car_only"])
    axes[i, 2].set_title("Без фона")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 13. Сохранение PNG с прозрачным фоном

In [ ]:
IMG_PATH = r"D:\ML_2\Carvana\train\00087a6bd4dc_01.jpg"
OUT_DIR  = r"D:\ML_2\Carvana\results"
os.makedirs(OUT_DIR, exist_ok=True)

name = os.path.splitext(os.path.basename(IMG_PATH))[0]
d    = get_all_masks(IMG_PATH, use_tta=True)
res  = ensemble(d, FINAL_CS, FINAL_WEIGHTS, FINAL_THRESHOLD, FINAL_MIN_AREA)

# Бинарная маска
mask_path = os.path.join(OUT_DIR, f"{name}_mask.png")
cv2.imwrite(mask_path, res["mask"])
print(f"Маска: {mask_path}")

# PNG с прозрачным фоном
rgba_path = os.path.join(OUT_DIR, f"{name}_no_bg.png")
rgba = np.dstack([d["img_rgb"], res["mask"]])
Image.fromarray(rgba.astype(np.uint8), mode="RGBA").save(rgba_path)
print(f"PNG с прозрачностью: {rgba_path}")